# DSE 200 — Day 3 Lab
## EDA, Clustering, Who Comes Back?, Classification

**Chunks 6.1, 6.2, the group challenge, and 7 · agent on — explain every cell**

Section numbers match the slides.

| Cell | What to do |
| --- | --- |
| `▶ DEMO` | Already written. Run it while Rick talks. |
| `🔍 READ` | **Predict before running.** Commit to an answer. |
| `✍️ YOU TRY` | Has a `# TODO`. Yours. |
| `✅ CHECKPOINT` | Run it. It tells you whether you are right. |

**Open the table of contents** (☰, left edge). *Runtime → Run all* if you get lost — every section rebuilds its own data.

In [ ]:
# ▶ SETUP — run first. Finds the data and defines check_read().
import os, textwrap
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

CANDIDATES = ["customers.csv", "/content/customers.csv", "data/customers.csv", "../customers.csv"]
PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)
if PATH is None:  # opened from GitHub: Colab brought the notebook but not the data
    try:
        import urllib.request
        urllib.request.urlretrieve("https://raw.githubusercontent.com/DSE200-2026/DSE200/main/customers.csv", "customers.csv")
        PATH = "customers.csv"
        print("downloaded customers.csv from the course repo")
    except Exception as e:
        print("could not download customers.csv:", e)
assert PATH, ("customers.csv not found -- drag it into the Files pane (folder icon, left), "
              "or run: from google.colab import files; files.upload()")

df = pd.read_csv(PATH)
print(f"df {df.shape}  from {PATH}")

FEATURES = ["n_orders", "total_spend", "avg_order_value",
            "days_since_last_order", "tenure_days", "n_categories"]

def clean(d):
    """The cleaning decisions from 6.7, in one place, so later sections start from the same table."""
    c = d.dropna(subset=FEATURES).copy()
    c = c[(c.total_spend >= 0) & (c.n_orders > 0) & (c.tenure_days <= 3650)].copy()
    c["avg_order_value"] = (c.total_spend / c.n_orders).round(2)   # recompute; 40 rows disagreed
    return c

_READ = {
"6.4": ("About 0.0.",
        "y = x**2 is a PERFECT relationship -- y is completely determined by x.\n"
        "Correlation is ~0 anyway, because it only measures STRAIGHT-LINE association.\n"
        "The left half slopes down, the right half slopes up, and they cancel.\n"
        "Correlation 0 means 'no linear relationship', not 'no relationship'. Plot it."),
"6.5": ("n_orders, +0.37 -- then days_since_last_order, -0.32.",
        "Ranked by SIZE, ignoring sign: n_orders +0.37, days_since_last_order -0.32,\n"
        "n_categories +0.24, total_spend +0.23, tenure_days +0.03.\n"
        "The negative one is just as strong: the longer someone has been gone,\n"
        "the less likely they come back. Sort by absolute value or you will miss it."),
"6.8": ("n_orders stops counting. The clusters come from spend and days gone.",
        "Distance adds up raw differences. Spend runs to ~3,800 and days gone to 900,\n"
        "so a gap of 10 orders is worth less than a $20 difference in spend.\n"
        "n_orders -- the strongest single predictor of returning -- gets almost no say.\n"
        "Scaling puts every column in standard deviations, so each gets an equal vote.\n"
        "It does not tidy the answer. It CHANGES it."),
"7.4": ("You cannot tell -- they are all within 5 points of 'always no'.",
        "baseline 0.777, logistic 0.823, KNN 0.808, tree 0.825, forest 0.814.\n"
        "Accuracy is dominated by the 78% who do not return. A model could miss\n"
        "EVERY returning customer and still score 0.777. The number you need is in\n"
        "the confusion matrix: how many of the 419 returners did each model miss?"),
}

def check_read(tag, prediction):
    if not str(prediction).strip() or str(prediction).strip() in ("???", "?"):
        print("Write a prediction first. Committing to a wrong answer is how this works.")
        return
    ans, why = _READ[tag]
    print("your prediction:"); print(textwrap.indent(str(prediction), "    "))
    print("\nactual:");        print(textwrap.indent(ans, "    "))
    print("\n" + why)

print("check_read() and clean() ready.")

# EDA and Visualization

*Slides D03.02 · sections 6.1–6.7 · look at the data before you model it · exercise 6.7*

## 6.1 What EDA Is

**Asking the data questions until you trust it.** Not: run `describe()`, paste the output, move on.

| You ask | You are checking |
| --- | --- |
| what is in here? | shape, columns, types |
| what is missing? | completeness |
| what is impossible? | validity |
| what is lopsided? | distributions, balance |
| what moves together? | relationships |

Our question all day: **will this customer come back?** That is the `returned` column.

## 6.2 The First Five Minutes

The same four calls, every new file, forever.

In [ ]:
# ▶ DEMO
print(df.shape)
print(df.dtypes)
df.head()

In [ ]:
# ▶ DEMO -- read describe() like an auditor: look at every min and max
df.describe().round(1)

In [ ]:
# ▶ DEMO -- what is missing: per column, as a fraction, and PER ROW
print(df.isna().sum(), "\n")
print((df.isna().sum() / len(df)).round(3), "\n")
print("missing values per row:")
print(df.isna().sum(axis=1).value_counts())

That last table is the one people skip. **9,250 rows are complete, 600 are missing one thing, and 150 are missing four things at once.** Scattered gaps and a block of broken rows are different problems — you would impute the first and go and ask about the second.

## 6.3 Distributions

Centre, spread, shape — one variable at a time.

In [ ]:
# ▶ DEMO
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
df["total_spend"].hist(bins=50, ax=axes[0])
axes[0].set_title("total_spend -- a long right tail")
np.log1p(df["total_spend"].clip(lower=0)).hist(bins=50, ax=axes[1])
axes[1].set_title("log1p(total_spend) -- the same data, readable")
plt.show()

print("skew  ", round(df["total_spend"].skew(), 2))
print("mean  ", round(df["total_spend"].mean(), 2))
print("median", round(df["total_spend"].median(), 2))

**On skewed money the mean is not typical.** A few big spenders drag it up. Report the median next to it, or someone will plan around a customer who does not exist.

In [ ]:
# ▶ DEMO -- outliers: a boxplot and the IQR rule
df.boxplot(column="total_spend", by="channel", figsize=(6, 4))
plt.suptitle(""); plt.show()

q1, q3 = df["total_spend"].quantile([0.25, 0.75])
iqr = q3 - q1
outliers = df[df["total_spend"] > q3 + 1.5 * iqr]
print(f"Q1 {q1:.0f}  Q3 {q3:.0f}  fence {q3 + 1.5*iqr:.0f}  ->  {len(outliers)} customers above it")

The rule **finds** them. It does not tell you what they are: an error, a rare real event, or **the most interesting customer you have**. Deleting outliers by default throws away the third.

## 6.4 Relationships

Two at a time — and always plot as well as measure.

In [ ]:
# ▶ DEMO
df.plot.scatter(x="n_orders", y="total_spend", alpha=0.15, figsize=(6, 4))
plt.show()
print(df.corr(numeric_only=True)["total_spend"].sort_values(ascending=False).round(3))

In [ ]:
# ▶ DEMO -- the heatmap: target, redundancy, and anything at 0.95+ that should not be
import seaborn as sns
plt.figure(figsize=(7, 5))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.show()

In [ ]:
# 🔍 READ 6.4 — y is completely determined by x. What is the correlation?
#
#     x = np.linspace(-3, 3, 200)
#     y = x ** 2
#     np.corrcoef(x, y)[0, 1]
#
# Commit first.

prediction = "???"
check_read("6.4", prediction)

## 6.5 The Target

Look at it before anything else.

In [ ]:
# ▶ DEMO
print(df["returned"].value_counts())
print(df["returned"].value_counts(normalize=True).round(3))

**About 22% of customers return.** A model that always says "no" is about **78% accurate** — and useless. That is your baseline, and it means accuracy is now a bad metric. Hold that thought until 7.4.

In [ ]:
# 🔍 READ 6.5 — which feature is MOST strongly related to `returned`, and which way?
#
#     df.corr(numeric_only=True)["returned"]
#
# Commit to a feature and a sign first.

prediction = "???"
check_read("6.5", prediction)

In [ ]:
# ▶ DEMO -- sort by SIZE, keep the sign
r = df.corr(numeric_only=True)["returned"].drop("returned")
r.reindex(r.abs().sort_values(ascending=False).index).round(3)

**Now suppose a new column appeared, correlating 0.94 with `returned`.** That is almost never a discovery. It is a field recorded *after* the outcome, an undocumented derived column, or an ID that sorts by outcome — **the column contains the answer**. That is leakage, and it is what you hunt in today's DOJO.

## 6.6 Regression Review

Week 2's shape, and why it is the wrong shape for today.

In [ ]:
# ▶ DEMO -- a number from a number: fine
from sklearn.linear_model import LinearRegression
c = clean(df)
line = LinearRegression().fit(c[["n_orders"]], c["total_spend"])
print(f"slope {line.coef_[0]:.2f} dollars per order   R2 {line.score(c[['n_orders']], c['total_spend']):.3f}")

In [ ]:
# ▶ DEMO -- a yes/no from a number: the line does not know it is a yes/no
yesno = LinearRegression().fit(c[["days_since_last_order"]], c["returned"])
pred = yesno.predict(c[["days_since_last_order"]])
print(f"smallest prediction {pred.min():.3f}   largest {pred.max():.3f}")
print(f"{(pred < 0).sum()} customers get a NEGATIVE chance of returning")

A straight line happily predicts −0.58. That is not a yes, a no, or a probability. **Logistic regression** (7.3) bends the same line into an S-curve that stays between 0 and 1. Same `fit` / `predict` shape.

## 6.7 Your Turn — 12 minutes

Interrogate the customer file.

In [ ]:
# ✍️ YOU TRY 6.7
# 1. n_rows, n_incomplete_rows, n_broken_rows
#       incomplete = missing ANYTHING;  broken = missing 4 or more values
# 2. n_negative_spend, n_ancient
#       two impossible values: spend below 0, and tenure over 3650 days (ten years)
# 3. spend_mean, spend_median   -- of total_spend
# 4. baseline   -- accuracy of always guessing the majority class of `returned`
# 5. top3       -- the 3 features most strongly correlated with `returned`, BY SIZE (a set of names)

n_rows = n_incomplete_rows = n_broken_rows = 0   # TODO
n_negative_spend = n_ancient = 0                 # TODO
spend_mean = spend_median = 0.0                  # TODO
baseline = 0.0                                   # TODO
top3 = set()                                     # TODO

print(n_rows, n_incomplete_rows, n_broken_rows)
print(n_negative_spend, n_ancient)
print(round(spend_mean, 2), round(spend_median, 2))
print(round(baseline, 4), top3)

In [ ]:
# ✅ CHECKPOINT 6.7
assert n_rows == 10_000, n_rows
assert n_incomplete_rows == 750, n_incomplete_rows
assert n_broken_rows == 150, n_broken_rows
assert n_negative_spend == 30, n_negative_spend
assert n_ancient == 3, n_ancient
assert abs(spend_mean - 313.02) < 0.5, spend_mean
assert abs(spend_median - 225.62) < 0.5, spend_median
assert abs(baseline - 0.7768) < 0.001, baseline
assert top3 == {"n_orders", "days_since_last_order", "n_categories"}, top3
print("Correct.\n")
print(f"Mean spend ${spend_mean:.0f}, median ${spend_median:.0f}: the mean is {spend_mean/spend_median - 1:.0%} higher.")
print(f"Always guessing 'no' is {baseline:.1%} accurate. Every model today has to beat that.")

**Write your three sentences here** (double-click to edit) — to whoever owns this data:

1. One about what is missing or broken.
2. One about the target and what it means for how we measure success.
3. One about anything that looks too good to be true. *Keep this one for the DOJO.*

*(your answer)*

# Clustering

*Slides D03.03 · section 6.8 · no target, five lines, two ways it misleads*

## 6.8 KMeans

1. Pick **k** — you, not the algorithm.
2. Drop k centres at random.
3. Assign every row to its nearest centre.
4. Move each centre to the middle of what it caught. Repeat until nothing moves.

In [ ]:
# ▶ SETUP 6.8 — from cold
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
c = clean(df)
K = ["total_spend", "n_orders", "days_since_last_order"]
print(c[K].describe().loc[["min", "max"]])

In [ ]:
# ▶ DEMO -- scale, fit, read the new column
Xs = StandardScaler().fit_transform(c[K])
km = KMeans(n_clusters=4, random_state=42, n_init=10).fit(Xs)
c["segment"] = km.labels_

c.groupby("segment").agg(customers=("returned", "size"),
                         spend=("total_spend", "median"),
                         orders=("n_orders", "median"),
                         days_gone=("days_since_last_order", "median"),
                         return_rate=("returned", "mean")).round(2)

In [ ]:
# 🔍 READ 6.8 — what changes if we skip StandardScaler?
#
#     km_raw = KMeans(n_clusters=4, random_state=42, n_init=10).fit(c[K])
#
# total_spend runs to ~3,800, days_since_last_order to 900, n_orders to ~34.
# Which column loses its say? Commit first.

prediction = "???"
check_read("6.8", prediction)

In [ ]:
# ▶ DEMO -- see it
km_raw = KMeans(n_clusters=4, random_state=42, n_init=10).fit(c[K])
c.assign(raw=km_raw.labels_).groupby("raw").agg(customers=("returned", "size"),
                                                spend=("total_spend", "median"),
                                                orders=("n_orders", "median"),
                                                days_gone=("days_since_last_order", "median"),
                                                return_rate=("returned", "mean")).round(2)

In [ ]:
# ▶ DEMO -- k is your decision. The elbow is a suggestion.
inertia = [KMeans(n_clusters=k, random_state=42, n_init=10).fit(Xs).inertia_ for k in range(2, 9)]
plt.plot(range(2, 9), inertia, marker="o")
plt.xlabel("k"); plt.ylabel("inertia (within-cluster spread)")
plt.title("where does another cluster stop buying much?")
plt.show()

**The real test is not the elbow. Can you name them?** Look back at the scaled table: one segment has been gone for most of a year and almost never returns; one orders often, spends most, and comes back half the time. If you can say each segment in a sentence a colleague would recognise, you have a finding. If not, you have arithmetic.

# Who Comes Back?

*Slides D03.04 · section 6.9 · group challenge · 35 minutes, then 5 minutes each to the room*

**One claim about who returns, backed by the data, that is not luck.** Weeks 1 and 2 only — pandas, `groupby`, `crosstab`, charts, the shuffle test. **No models.** Your angle is on your card.

Present four things:

1. **One chart whose title is the finding** — sorted, starts at zero, units labelled
2. **Why it is not luck** — your shuffle-test number
3. **One thing wrong with the data**, and what you did about it
4. **Your bet:** *"the feature a model will rank highest is ___"*

In [ ]:
# ▶ SETUP 6.9 — from cold: the raw file AND the cleaned one with segments
c = clean(df)
Xs = StandardScaler().fit_transform(c[["total_spend", "n_orders", "days_since_last_order"]])
c["segment"] = KMeans(n_clusters=4, random_state=42, n_init=10).fit(Xs).labels_
print(f"df: {len(df)} raw rows    c: {len(c)} clean rows, with a segment column")

In [ ]:
# ▶ DEMO -- the shuffle test from Week 1, for any column
def shuffle_test(data, col, n=300):
    """How often does SHUFFLED `returned` produce a gap between groups as big as the real one?"""
    real = data.groupby(col)["returned"].mean()
    gap = real.max() - real.min()
    shuffled = []
    for i in range(n):
        fake = data["returned"].sample(frac=1, random_state=i).values
        rates = data.assign(fake=fake).groupby(col)["fake"].mean()
        shuffled.append(rates.max() - rates.min())
    share = np.mean([s >= gap for s in shuffled])
    print(f"{col}: our gap {gap:.3f}   shuffles at least as big: {share:.0%}")
    return share

shuffle_test(df, "channel")

**Reading it:** 0–1% of shuffles matching your gap — you have found something. More than 5% — you have not, yet. Say so; that is a result too. In Week 1 the city spread looked real and was noise.

Your group's work goes below. Add as many cells as you need.

In [ ]:
# your group's work starts here

# Classification with scikit-learn

*Slides D03.05 · sections 7.1–7.5 · four models, then learn to distrust them · exercise 7.5*

## 7.1 The Setup

`X` is what you know — a table. `y` is what you want — a column.

In [ ]:
# ▶ SETUP 7.1 — from cold
from sklearn.model_selection import train_test_split
c = clean(df)
X = c[FEATURES]
y = c["returned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(len(X_train), "train  ", len(X_test), "test")
print("positives:", round(y_train.mean(), 3), "train ", round(y_test.mean(), 3), "test")
print("rows in both halves:", len(set(X_train.index) & set(X_test.index)))   # must be 0

`stratify=y` keeps 22% positives in both halves. **If a row appeared in both, every number afterwards would be a lie** — not an obvious one, a flattering one.

## 7.2 The Protocol

Every model: build it with settings, `.fit()` it, `.predict()` / `.score()` with it. **And a baseline first.**

In [ ]:
# ▶ DEMO
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("always 'no':", round(dummy.score(X_test, y_test), 3))

## 7.3 Four Models

In [ ]:
# ▶ DEMO -- logistic regression: the line, bent into a probability
from sklearn.linear_model import LogisticRegression
logreg = LogisticRegression(max_iter=1000).fit(X_train, y_train)
print("accuracy", round(logreg.score(X_test, y_test), 3))
print("first five probabilities:", logreg.predict_proba(X_test)[:5, 1].round(2))
pd.Series(logreg.coef_[0], index=FEATURES).round(4)

**Careful reading `coef_`:** each number is "per one unit of this feature". `total_spend` is in dollars and `n_orders` in orders, so their sizes are not comparable. The **signs** are: more orders → more likely back; more days gone → less likely.

In [ ]:
# ▶ DEMO -- nearest neighbour: scale first, and fit the scaler on TRAIN only
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
scaler = StandardScaler().fit(X_train)
knn = KNeighborsClassifier(n_neighbors=15).fit(scaler.transform(X_train), y_train)
print("accuracy", round(knn.score(scaler.transform(X_test), y_test), 3))

In [ ]:
# ▶ DEMO -- how a tree picks a split: Gini on the loan example from the slides
def gini(repay, default):         # 0 = pure, 0.5 = a coin toss
    n = repay + default
    return 1 - (repay / n) ** 2 - (default / n) ** 2

root = gini(25, 15)                       # all 40 customers
left, right = gini(8, 10), gini(17, 5)    # split on income > $50K
after = (18 / 40) * left + (22 / 40) * right
print(f"before {root:.3f}   after {after:.3f}   -- the drop is why this split wins")

In [ ]:
# ▶ DEMO -- a decision tree on our customers. You can read it.
from sklearn.tree import DecisionTreeClassifier, plot_tree
tree = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train)
print("accuracy", round(tree.score(X_test, y_test), 3))

plt.figure(figsize=(14, 6))
plot_tree(tree, feature_names=FEATURES, max_depth=2, filled=True, fontsize=8)
plt.show()

In [ ]:
# ▶ DEMO -- and its flaw: train and test, as the tree grows
for depth in [2, 4, 8, None]:
    t = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train, y_train)
    print(f"max_depth {str(depth):>4}   train {t.score(X_train, y_train):.3f}   test {t.score(X_test, y_test):.3f}")

With no limit the tree scores **1.000** on rows it has seen and worse than the depth-4 tree on rows it has not. **The gap is the overfitting.** `max_depth` is the leash.

In [ ]:
# ▶ DEMO -- random forest, and CHECK YOUR BETS from the challenge
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=300, random_state=42).fit(X_train, y_train)
print("accuracy", round(rf.score(X_test, y_test), 3))
pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False).round(3)

Put that list next to the bets on the board. Then look hard at **`tenure_days`**: its correlation with `returned` is 0.03, yet the forest ranks it near the top.

That is not a hidden truth. A continuous column offers the trees hundreds of cut points, and on training data some of them always look useful. **Importance means "the model leaned on it" — not "it matters", and certainly not "it causes returning".** Week 4's interpretability chunk is about exactly this gap.

## 7.4 Accuracy Lies

In [ ]:
# 🔍 READ 7.4 — which model is best?
#
#     baseline (always "no")     0.777
#     logistic regression        0.823
#     nearest neighbour          0.808
#     decision tree              0.825
#     random forest              0.814
#
# Commit to an answer -- and say what you would need to know to be sure.

prediction = "???"
check_read("7.4", prediction)

In [ ]:
# ▶ DEMO -- four numbers instead of one
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
ConfusionMatrixDisplay.from_estimator(logreg, X_test, y_test)
plt.show()

tn, fp, fn, tp = confusion_matrix(y_test, logreg.predict(X_test)).ravel()
print(f"returning customers in the test set: {fn + tp}")
print(f"  caught {tp}, MISSED {fn}")
print(f"expected back but never returned: {fp}")

**The logistic model's 0.82 is real. It also missed about six in ten of the customers we care about.** What to do about that — precision, recall, moving the threshold, the 22% itself — is the next chunk. Bring your missed-returner count with you.

## 7.5 Your Turn — 12 minutes

Four models, one table.

In [ ]:
# ✍️ YOU TRY 7.5
# 1. results -- a dict: model name -> (accuracy, returners_missed) on the TEST set,
#    for "logistic", "knn", "tree", "forest". Reuse logreg, knn (+ scaler), tree, rf from 7.3.
#    returners_missed = actual 1, predicted 0. confusion_matrix(...).ravel() gives tn, fp, fn, tp.
# 2. gap -- train accuracy minus test accuracy for an UNLIMITED tree (max_depth=None, random_state=42)

results = {}   # TODO
gap = 0.0      # TODO

for name, (acc, missed) in results.items():
    print(f"{name:9} accuracy {acc:.3f}   missed {missed}")
print("unlimited-tree gap:", round(gap, 3))

In [ ]:
# ✅ CHECKPOINT 7.5
assert set(results) == {"logistic", "knn", "tree", "forest"}, sorted(results)
for name, (acc, missed) in results.items():
    assert 0.79 < acc < 0.85, (name, acc)            # all beat 0.777 -- but not by much
    assert 180 < missed < 290, (name, missed)        # and all miss a lot of the 419
assert abs(results["logistic"][0] - 0.823) < 0.01, results["logistic"]
assert results["logistic"][1] == 255, results["logistic"]
assert gap > 0.2, gap
print("Correct.\n")
best = max(results, key=lambda k: results[k][0])
fewest = min(results, key=lambda k: results[k][1])
print(f"Highest accuracy: {best}.  Fewest returners missed: {fewest}.")
if best != fewest:
    print("Different models -- accuracy was answering the wrong question.")
else:
    print(f"Same model this time. It still missed {results[best][1]} of the 419 who came back,")
    print("and the accuracy column never told you.")

**Write your two sentences here** (double-click to edit):

> Which model would you carry into the next hour? One sentence naming it and the evidence. One naming what it still gets wrong — in returning customers missed, not in accuracy.

*(your answer)*

---

**Then compare:** do `logreg.coef_` and `rf.feature_importances_` agree on what matters? Where they disagree, which one would you trust — and why?

## Wrap

You can now:

- Run the first five minutes on any file, and read missingness **per row** as well as per column
- Tell skew from symmetry, and report the median when the mean stops being typical
- Read correlations by size **and** sign, and be suspicious of anything too good
- Cluster with KMeans — scale first, choose k, name the segments or drop them
- Prove a group difference is not luck with the shuffle test
- Split, baseline, and fit four classifiers with the same four lines
- Spot overfitting in the train/test gap
- Read a confusion matrix, and say what accuracy was hiding

**Next:** feature engineering and class imbalance — then the audit-the-agent DOJO.